1. divide my data to 5 year non-overlapping window in age (people 20-24, 25-29, 30-34 etc.)

2. Per ROI, and per age window, I want to find the slope of birth year (b1)
the regression model is:

for age_window:

GMV_ROI_1 ~ b1*birth_year

3. after calculating all b1 for all age windows, I want to present them in a plot (y axis is GMV, x axisis birth year).

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

import pandas as pd
import statsmodels.formula.api as smf
from tqdm import tqdm

In [ ]:
combined_df = pd.read_pickle('/home/gaia/Projects/legacy_data/combined_gm_volumes.pkl')
 
# keep only classification_label=1 and snbb
volumes = combined_df[(combined_df['classification_label'] == 1) | (combined_df['source'] == 'snbb')]


In [ ]:

# divide volumes into age bins according to bin size
bin_size = 5
volumes['age_bin'] = pd.cut(volumes['age_in_years'], bins=np.arange(15, 85, bin_size))

print(f"amount of scans in each bin:")
print(volumes['age_bin'].value_counts().sort_index())

bin_list = volumes['age_bin'].cat.categories.tolist()


In [ ]:
# add region name according to the atlas 
atlas_csv = pd.read_csv("/home/gaia/Projects/legacy_data/my_master/space-MNI152_atlas-schaefer2018tian2020_res-1mm_den-400_div-7networks_dseg.csv")
import pandas as pd
import statsmodels.formula.api as smf
from tqdm import tqdm

In [ ]:
# initiate a df with the columns bin, region_label, birth_year_coef, birth_year_t, birth_year_p
coef_df = pd.DataFrame(columns=['age_bin', 'region_label', 'variable', 'coef', 't', 'p', 'fdr_p', 'region_name'])

for bin in volumes['age_bin'].unique().dropna():
    print(f"age bin: {bin}, number of scans: {volumes[volumes['age_bin'] == bin].shape[0]}")

    min_age = bin.left
    max_age = bin.right

    # volumes df
    df_bin = volumes[(volumes['age_in_years'] >= min_age) & (volumes['age_in_years'] < max_age)]

    # metadata df
    # remove duplicates based on subject_id
    metadata = volumes.drop_duplicates(subset=['subject_id'])
    print(f"shape of {min_age} - {max_age} years old metadata after removing duplicates: {metadata.shape}")



    # create a multiple regression model for each ROI
    results = []

    # Loop over regions (ROI-level analyses)
    for roi, df_roi in tqdm(df_bin.groupby('region_label')):
        # Fit model with birth_year as a continuous predictor
        model = smf.ols(
            'volume_mm3 ~ birth_year + C(sex) + tiv + age_in_years',
            data=df_roi
        ).fit()
        
        # Collect stats for each variable
        for var in model.params.index:
            results.append({
                'region_label': roi,
                'variable': var,
                'coef': model.params[var],
                't': model.tvalues[var],
                'p': model.pvalues[var]
            })

    results_df = pd.DataFrame(results)

    results_df = results_df.merge(
        atlas_csv[['index', 'name']],     # only keep relevant columns
        how='left',                      # keep all rows from results_df
        left_on='region_label',          # column in results_df
        right_on='index'                 # matching column in atlas_csv
    )

    # rename and clean up
    results_df.rename(columns={'name': 'region_name'}, inplace=True)
    results_df.drop(columns='index', inplace=True)

    # --- Multiple comparison correction (using results_df) ---
    from statsmodels.stats.multitest import multipletests

    cov_of_interest = 'birth_year'
    mask = results_df['variable'] == cov_of_interest
    _, fdr_p, _, _ = multipletests(results_df.loc[mask, 'p'], method='fdr_bh')
    results_df.loc[mask, 'fdr_p'] = fdr_p

    # sort for inspection
    results_df = results_df.sort_values(by='fdr_p')
    print(results_df.head())


    # make sure fdr_p is float 
    results_df['fdr_p'] = results_df['fdr_p'].astype(float)

    results_df['age_bin'] = bin
    coef_df = pd.concat([coef_df, results_df], ignore_index=True)

    # save the rows from results_df where fdr_p < 0.05
    significant_results_df = results_df[results_df['fdr_p'] < 0.05].copy()

    # increase in volume (t>0)
    print(significant_results_df.loc[significant_results_df['t'] > 0, 'region_name'].tolist())

    # decrease
    print(significant_results_df.loc[significant_results_df['t'] < 0, 'region_name'].tolist())

In [ ]:
birth_year_coef_df = coef_df[coef_df['variable'] == 'birth_year'].copy()


In [ ]:
birth_year_coef_df.to_csv(f"/home/gaia/Projects/legacy_data/legacy_pipe/data/interim/birth_year_coef_df_age_bins_size_{bin_size}.csv", index=False)

# Visualization

Trends per ROI 

In [ ]:
birth_year_coef_df['mid_age_bin'] = birth_year_coef_df['age_bin'].apply(lambda x: x.mid)

# plot coef vs age bin for a specific region
# all of them, list from 1 to 454
roi_list = list(range(1, 455))
# roi_list = [421,422,448,449,453,426]

for region_of_interest in roi_list:
    region_data = birth_year_coef_df[birth_year_coef_df['region_label'] == region_of_interest]
    plt.figure(figsize=(5, 3))
    sns.lineplot(
        data=region_data,
        x='mid_age_bin',
        y='coef',
        marker='o'
    )
    plt.title(f'Birth Year Coefficient vs Age Bin for Region {region_of_interest}')
    plt.xlabel('Age Bin')
    plt.ylabel('Birth Year Coefficient')
    plt.xticks(rotation=45)
    plt.grid()
    plt.tight_layout()
    plt.show()

Linear regression and on to the brain 
The slope of birth year in each age window 
the effect of birth year in each age window

In [ ]:
# # visualization of significant results on the brain, birth_year

# import numpy as np
# import nibabel as nib
# from nilearn import plotting
# from nilearn.image import new_img_like
# from nilearn.datasets import load_mni152_template # for background image

# # --- 1. Load Your Atlas ---
# atlas_file_path = '/media/storage/MATLAB_atlases-20250612T134418Z-1-001/MATLAB_atlases/schaefer2018tian2020_400_7.nii'
# atlas_img = nib.load(atlas_file_path)
# atlas_labels = atlas_img.get_fdata()


# # --- 2. Extract significant T-values from your results_df ---
# significant_rois_data = birth_year_coef_df[(birth_year_coef_df['fdr_p'] < 0.05)][['region_label', 'coef']].values.tolist()
# # Create a dictionary for quick lookup of T-values by ROI label
# t_value_map = {label: t_val for label, t_val in significant_rois_data}
# path_to_t_map = f"/home/gaia/Projects/legacy_data/legacy_pipe/data/interim/t_value_map_{min_age}_to_{max_age}.csv"
# (pd.Series(t_value_map)).to_csv(path_to_t_map)


# # --- 3. Create the full statistical map ---

# # 1. Initialize the statistical map array to zeros.
# # This ensures that all non-significant ROIs (and the background 0) are set to T=0.
# stat_map_data = np.zeros_like(atlas_labels, dtype=float)

# # 2. Iterate over the significant ROIs and map their T-value.
# print(f"Mapping {len(significant_rois_data)} significant T-values...")
# for label, t_value in t_value_map.items():
#     # Find all voxels in the atlas that match the current ROI label and set their value in the stat_map_data to the T-value
#     stat_map_data[atlas_labels == label] = t_value

# # 3. Create the final NIfTI image for visualization.
# # This ensures the T-map has the same spatial dimensions and coordinates as your atlas.
# t_map_img = new_img_like(atlas_img, stat_map_data)
# print("Created T-map image for visualization.")


# # --- 4. Load background template ---
# bg_img = load_mni152_template()

# # --- 5. Plot the T-map ---
# custom_cut_coords = (6,6,6)

# # Plot the T-map using the mosaic display mode
# plotting.plot_stat_map(
#     stat_map_img=t_map_img,
#     bg_img=bg_img,
#     # title=f"ROI T-Values",
#     cmap='RdBu_r',
#     symmetric_cbar=True,
#     threshold=0.01,
#     display_mode='mosaic',
#     cut_coords=custom_cut_coords,
#     colorbar=True
# )


# # Display the plot
# plotting.show()